In [1]:
%load_ext autoreload 
%autoreload 2

In [2]:
from typing import Tuple, List

In [5]:
import torch
import numpy as np
import random
import os
from matplotlib import pyplot as plt

def set_seed(seed: int = 42) -> None:
    """Sets the random seed for reproducibility across PyTorch, NumPy, and Python's random module."""
    os.environ['PYTHONHASHSEED'] = str(seed)  # For Python's hash seed
    torch.manual_seed(seed)  # For PyTorch's CPU and CUDA RNGs
    torch.cuda.manual_seed(seed)  # For CUDA devices specifically
    torch.cuda.manual_seed_all(seed) # For all CUDA devices if multiple are used
    np.random.seed(seed)  # For NumPy's random number generator
    random.seed(seed)  # For Python's built-in random module

    # For deterministic algorithms in PyTorch (optional, but recommended for full reproducibility)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Example usage:
set_seed(123)

Step 1. Loading Data

In [6]:
import sys
import os

# Add project root (parent of "src") to Python path
sys.path.append(os.path.abspath(".."))

from src.data_loader import load_OhioT1DM_patient_split

Ohio_PATH = "..\data\\raw\\"
X_train, X_val, X_test, y_reg_train, y_reg_val, y_reg_test, y_clf_train, y_clf_val, y_clf_test = load_OhioT1DM_patient_split(path=Ohio_PATH, look_back=128)

In [7]:
LBW = X_train.shape[1]
PH = y_reg_train.shape[1]

Step 2. Define Model

In [8]:
import torch.nn as nn 
import torch.optim as optim 
import torch.nn.functional as F 
from torch import Tensor
from torch.utils.data import Dataset, DataLoader, RandomSampler
from torch.amp import GradScaler, autocast
import mlflow
import tqdm


mlflow.set_tracking_uri(uri="http://127.0.0.1:8080")

In [9]:
import math

def conv1d_output_length(L_in, kernel_size, stride=2, padding=1, dilation=1):
    """
    Compute the output length of a 1D convolution layer.

    Parameters
    ----------
    L_in : int
        Input length (sequence length).
    kernel_size : int
        Size of the convolution kernel.
    stride : int, default=1
        Convolution stride.
    padding : int, default=0
        Padding on both sides.
    dilation : int, default=1
        Dilation factor.

    Returns
    -------
    L_out : int
        Output length after the convolution.
    """
    return math.floor((L_in + 2*padding - dilation*(kernel_size - 1) - 1) / stride + 1)

In [10]:
# -----------------------
# Generator
# -----------------------
class Generator1D(nn.Module):
    def __init__(self, z_dim=16, signal_length=128, base_channels=64):
        super().__init__()
        self.z_dim = z_dim
        self.signal_length = signal_length

        # Project latent vector into a feature map
        self.fc = nn.Linear(z_dim, base_channels * (signal_length // 16))

        # Upsample progressively
        self.net = nn.Sequential(
            nn.ConvTranspose1d(base_channels, base_channels // 2, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm1d(base_channels // 2),
            nn.ReLU(True),

            nn.ConvTranspose1d(base_channels // 2, base_channels // 4, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm1d(base_channels // 4),
            nn.ReLU(True),

            nn.ConvTranspose1d(base_channels // 4, base_channels // 8, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm1d(base_channels // 8),
            nn.ReLU(True),

            nn.ConvTranspose1d(base_channels // 8, 1, kernel_size=4, stride=2, padding=1),
        )

    def forward(self, z):
        x = self.fc(z)  # (B, base_channels * L/16)
        x = x.view(x.size(0), -1, self.signal_length // 16)  # (B, C, L/16)
        return self.net(x)  # (B, 1, L)

    def sample(self, num_samples, device): 
        with torch.no_grad(): 
            z = torch.randn(num_samples, self.z_dim, device=device) 
            samples = self(z) 
            
        return samples 


# -----------------------
# Discriminator
# -----------------------
class Discriminator1D(nn.Module):
    def __init__(self, signal_length=128, base_channels=64):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv1d(1, base_channels // 8, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv1d(base_channels // 8, base_channels // 4, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm1d(base_channels // 4),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv1d(base_channels // 4, base_channels // 2, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm1d(base_channels // 2),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv1d(base_channels // 2, base_channels, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm1d(base_channels),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Flatten(),
            nn.Linear(base_channels * (signal_length // 16), 1), 
        )

    def forward(self, x):
        return self.net(x)  # raw logits (B,1)

In [11]:
# custom weights initialization called on ``netG`` and ``netD``
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

In [12]:
import torch
import torch.nn.functional as F

def d_loss_fn(real_logits, fake_logits):
    """
    Discriminator loss: wants real=1, fake=0
    """
    real_labels = torch.ones_like(real_logits)
    fake_labels = torch.zeros_like(fake_logits)
    real_loss = F.binary_cross_entropy_with_logits(real_logits, real_labels)
    fake_loss = F.binary_cross_entropy_with_logits(fake_logits, fake_labels)
    return real_loss + fake_loss

def g_loss_fn(fake_logits):
    """
    Generator loss: wants fake=1 (fool the discriminator)
    """
    real_labels = torch.ones_like(fake_logits)
    return F.binary_cross_entropy_with_logits(fake_logits, real_labels)


Step 3. Data Loader

In [13]:
class CustomDataset(Dataset):
    def __init__(self, X, y, device, classification=True):
        self.X = torch.tensor(X, dtype=torch.float32, device=device)
        if y is not None:
            if classification:
                self.y = torch.tensor(y, dtype=torch.long, device=device)
            else:
                self.y = torch.tensor(y, dtype=torch.float32, device=device)
        else:
            self.y = None

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        return self.X[idx]

Step 4. Trainer

In [14]:
# -----------------------
# DCGAN Trainer
# -----------------------
class DCGANTrainer:
    def __init__(self, generator:Generator1D, discriminator:Discriminator1D, g_optimizer, d_optimizer,
                 device="cpu", batch_size=32, fp16=False, experiment_name="DCGAN-Experiment"):
        self.G = generator.to(device)
        self.D = discriminator.to(device)
        self.g_optimizer = g_optimizer
        self.d_optimizer = d_optimizer
        self.fixed_noize = torch.randn(8, self.G.z_dim)
        self.device = device
        self.batch_size = batch_size
        self.fp16 = fp16
        self.scaler = GradScaler('cuda', enabled=fp16)
        self.experiment_name = experiment_name

    def _train_step(self, real_data, z_dim, epoch:int):
        real_data = real_data.to(self.device).unsqueeze(1)  # (B,1,L)
        batch_size = real_data.size(0)

        # Labels
        real_labels = torch.ones(batch_size, 1, device=self.device)
        fake_labels = torch.zeros(batch_size, 1, device=self.device)

        # =============== Train Discriminator ===============


        self.d_optimizer.zero_grad()
        if epoch  % self.g_d_train_ratio == 0 :
            z = torch.randn(batch_size, z_dim, device=self.device)
            fake_data = self.G(z)
            with autocast(device_type="cuda", dtype=torch.float16, enabled=self.fp16):
                real_pred = self.D(real_data)
                fake_pred = self.D(fake_data.detach())

                d_loss_real = F.binary_cross_entropy_with_logits(real_pred, real_labels)
                d_loss_fake = F.binary_cross_entropy_with_logits(fake_pred, fake_labels)
                d_loss = d_loss_real + d_loss_fake

            if self.fp16:
                self.scaler.scale(d_loss).backward()
                self.scaler.step(self.d_optimizer)
            else:
                d_loss.backward()
                self.d_optimizer.step()
        else: 
            d_loss = torch.tensor([0])

        # =============== Train Generator ===============
        self.g_optimizer.zero_grad()
        z = torch.randn(batch_size, z_dim, device=self.device)
        fake_data = self.G(z)
        with autocast(device_type="cuda", dtype=torch.float16, enabled=self.fp16):
            fake_pred = self.D(fake_data)
            g_loss = F.binary_cross_entropy_with_logits(fake_pred, real_labels)

        if self.fp16:
            self.scaler.scale(g_loss).backward()
            self.scaler.step(self.g_optimizer)
            self.scaler.update()
        else:
            g_loss.backward()
            self.g_optimizer.step()

        return d_loss.item(), g_loss.item(), fake_data.detach().cpu()

    def _log_samples(self, samples, epoch, normalized:bool, n_samples=8):
        samples = samples[:n_samples].squeeze(1)
        if normalized:
            samples = self.denormalize(samples).numpy()
        fig, axes = plt.subplots(nrows=n_samples, ncols=1, figsize=(6, n_samples*1.5))
        for i in range(n_samples):
            axes[i].plot(samples[i], color="blue")
        plt.tight_layout()
        fname = f"dcgan_samples_epoch_{epoch}.png"
        plt.savefig(fname)
        plt.close(fig)
        mlflow.log_artifact(fname)
        os.remove(fname)
        
    def normalize(self, X_train):
        self.x_max = X_train.max()
        self.x_min = X_train.min()
        return 2 * (X_train - self.x_min) / (self.x_max - self.x_min) - 1
    
    def denormalize(self, X_train):
        return (X_train + 1) / 2 * (self.x_max - self.x_min) + self.x_min
    
    def fit(self, X_train, z_dim=16, epochs=20, normalize:bool = True, g_d_train_ratio:int = 4):
        train_loader = DataLoader(CustomDataset(X_train, None, self.device),
                                  sampler=RandomSampler(X_train),
                                  batch_size=self.batch_size,
                                  drop_last=True, )
        self.g_d_train_ratio = g_d_train_ratio                         
        if normalize: 
            X_train = self.normalize(X_train)
        mlflow.set_experiment(self.experiment_name)
        with mlflow.start_run():
            mlflow.log_params({
                "z_dim": z_dim,
                "batch_size": self.batch_size,
                "epochs": epochs
            })

            history = {"d_loss": [], "g_loss": []}

            for epoch in range(1, epochs+1):
                self.d_losses, self.g_losses = [], []
                for X in tqdm.tqdm(train_loader, desc=f"Epoch {epoch} [Train]"):
                    d_loss, g_loss, fake_samples = self._train_step(X, z_dim, epoch)
                    self.d_losses.append(d_loss)
                    self.g_losses.append(g_loss)

                if epoch  % self.g_d_train_ratio == 0 :
                    avg_d = np.mean(self.d_losses)
                    history["d_loss"].append(avg_d)
                    mlflow.log_metric("d_loss", avg_d, step=epoch)
                
                avg_g = np.mean(self.g_losses)
                history["g_loss"].append(avg_g)
                mlflow.log_metric("g_loss", avg_g, step=epoch)

                self._log_samples(fake_samples, epoch, normalized=normalize)

                # Save checkpoints
            torch.save(self.G.state_dict(), f"../models/dcgan_G.pt")
            mlflow.log_artifact("../models/dcgan_G.pt")
            torch.save(self.D.state_dict(), f"../models/dcgan_D.pt")
            mlflow.log_artifact("../models/dcgan_D.pt")

            return history

Step 5. Train

In [15]:
z_dim = 16
signal_length = 128

G = Generator1D(z_dim=z_dim, signal_length=signal_length)
D = Discriminator1D(signal_length=signal_length)

G.apply(weights_init)
D.apply(weights_init)

g_optimizer = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
d_optimizer = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))

trainer = DCGANTrainer(
    generator=G,
    discriminator=D,
    g_optimizer=g_optimizer,
    d_optimizer=d_optimizer,
    device="cuda" if torch.cuda.is_available() else "cpu",
    batch_size=128,
    fp16=False,
    experiment_name="DCGAN1D-CGM", 
)

history = trainer.fit(X_train, z_dim=z_dim, epochs=300, g_d_train_ratio = 1, normalize=False)


Epoch 300 [Train]: 100%|██████████| 507/507 [00:06<00:00, 77.70it/s]


🏃 View run gregarious-trout-468 at: http://127.0.0.1:8080/#/experiments/630093492432297283/runs/4f37e065b2274fffb108e73943eee308
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/630093492432297283


In [16]:
trainer.d_losses

[0.5233520269393921,
 0.6256739497184753,
 0.4874364137649536,
 0.641742467880249,
 0.16541139781475067,
 0.4822969138622284,
 0.8468905687332153,
 0.39403778314590454,
 0.4723339080810547,
 0.9760451316833496,
 0.3772466480731964,
 0.7150182127952576,
 0.1896921545267105,
 0.8389180302619934,
 0.2725076675415039,
 0.3206383287906647,
 0.41629210114479065,
 0.23205827176570892,
 0.5971229672431946,
 0.4842165410518646,
 0.6205892562866211,
 0.5595810413360596,
 0.421627938747406,
 0.3094954490661621,
 0.5804745554924011,
 0.5030471086502075,
 0.7742283344268799,
 0.7572113275527954,
 0.18320481479167938,
 0.3670997619628906,
 0.5684816837310791,
 0.5280317664146423,
 0.23510615527629852,
 0.2539738714694977,
 0.6443513631820679,
 0.6212351322174072,
 0.42073947191238403,
 0.6328854560852051,
 0.520781397819519,
 0.6505178809165955,
 0.5466623902320862,
 0.5135265588760376,
 0.6648398041725159,
 0.18769824504852295,
 0.24381878972053528,
 0.42077702283859253,
 0.23231419920921326,
 0.49

In [17]:
mlflow.end_run()

In [18]:
X_train.min()

40.0